# Notebook 1: Supervised Fine-Tuning (SFT)

## What we're doing
We fine-tune **Qwen2.5-0.5B-Instruct** on cybersecurity Q&A pairs using standard cross-entropy loss.
The model learns to imitate expert analyst responses to security questions.

## Task
**Threat knowledge Q&A** — given a cybersecurity question, generate an accurate, expert-level answer.

## Dataset
- Primary: `haonan-li/secqa` (HuggingFace) — multiple-choice cybersec questions
- Fallback: built-in synthetic Q&A pairs (no internet required)

## How SFT works
```
for each (prompt, response) pair:
    tokens = tokenize(prompt + response)
    loss   = CrossEntropy(model(tokens), tokens)  # only on response tokens
    loss.backward()
    optimizer.step()
```
The model is trained to predict each response token given the prompt + previous tokens.
Loss is masked so we only backprop through response tokens, not the prompt.

## CPU config
- batch_size=1, gradient_accumulation=8 (effective batch=8)
- max_seq_len=128, fp32, 1 epoch
- ~200 training samples → expect ~30–60 min on CPU

In [ ]:
# ── Cell 1: Install dependencies ────────────────────────────────────────────
import subprocess, sys
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers>=4.40', 'trl>=0.8', 'datasets', 'accelerate', 'torch'
])
print('Dependencies installed.')

In [ ]:
# ── Cell 2: Imports & project path setup ────────────────────────────────────
import os, sys, json, random, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Add project root to path so we can import shared utilities
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import torch
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig

from data.loader import load_sft_dataset, train_val_split
from shared.model_utils import (
    load_model_and_tokenizer, CPU_TRAIN_CONFIG,
    LossTracker, save_model, evaluate_sft
)

os.environ['CUDA_VISIBLE_DEVICES'] = ''
random.seed(42)
torch.manual_seed(42)
print(f'PyTorch {torch.__version__} | device: cpu')

In [ ]:
# ── Cell 3: Load & inspect dataset ──────────────────────────────────────────
MAX_SAMPLES = 200   # keep small for CPU

raw_data = load_sft_dataset(max_samples=MAX_SAMPLES)
train_data, val_data = train_val_split(raw_data, val_ratio=0.1)

print(f'\nTrain: {len(train_data)} | Val: {len(val_data)}')
print('\n--- Sample (train[0]) ---')
print('PROMPT:\n', train_data[0]['prompt'])
print('\nRESPONSE:\n', train_data[0]['response'])

In [ ]:
# ── Cell 4: Format data for SFT ─────────────────────────────────────────────
# TRL SFTTrainer expects a 'text' column with the full prompt+response concat.
# We add a special separator so loss masking can work correctly.
# With SFTTrainer, we pass formatting_func and it handles the masking.

def format_for_sft(example):
    """
    Concat prompt + response with EOS.
    TRL SFTTrainer applies loss only on tokens AFTER the prompt when
    we use the chat template or a formatting_func.
    """
    return {
        'text': example['prompt'] + ' ' + example['response'] + ' <|endoftext|>'
    }

train_hf = Dataset.from_list([format_for_sft(d) for d in train_data])
val_hf   = Dataset.from_list([format_for_sft(d) for d in val_data])

print('Dataset columns:', train_hf.column_names)
print('Sample text[:120]:', train_hf[0]['text'][:120])

In [ ]:
# ── Cell 5: Load model ───────────────────────────────────────────────────────
# This will download ~1GB on first run. Subsequent runs use cache.
model, tokenizer = load_model_and_tokenizer()

# Inspect model structure
print('\nModel architecture summary:')
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'  Total params:     {total_params/1e6:.1f}M')
print(f'  Trainable params: {trainable/1e6:.1f}M')
print(f'  Layers: {model.config.num_hidden_layers}')
print(f'  Hidden size: {model.config.hidden_size}')
print(f'  Vocab size: {model.config.vocab_size}')

In [ ]:
# ── Cell 6: Baseline inference BEFORE training ───────────────────────────────
from shared.model_utils import generate_response

test_prompt = (
    'You are a cybersecurity expert. Answer concisely.\n\n'
    'Question: What is SQL injection?\n\nAnswer:'
)

print('=== BEFORE SFT ===')
response_before = generate_response(model, tokenizer, test_prompt, max_new_tokens=80)
print(response_before)

In [ ]:
# ── Cell 7: Configure & run SFT training ─────────────────────────────────────
OUTPUT_DIR = str(PROJECT_ROOT / 'models' / 'sft_checkpoint')

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    # ── CPU-safe settings ──
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    max_seq_length=128,
    num_train_epochs=1,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    # ── Logging ──
    logging_steps=5,
    save_steps=100,
    eval_strategy='steps',
    eval_steps=25,
    load_best_model_at_end=False,
    # ── Precision ──
    fp16=False,
    bf16=False,
    # ── Misc ──
    report_to='none',
    dataloader_num_workers=0,  # Windows-safe
    dataset_text_field='text',
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=sft_config,
    train_dataset=train_hf,
    eval_dataset=val_hf,
)

print('Starting SFT training...')
print(f'  Steps: {len(train_hf) // (1 * 8)}')
print(f'  Effective batch size: {1 * 8}')
print('  (Each step processes 1 sample; loss accumulated over 8 steps before update)\n')

train_result = trainer.train()

print('\n=== Training complete ===')
print(f'  Final loss:       {train_result.training_loss:.4f}')
print(f'  Training time:    {train_result.metrics["train_runtime"]:.0f}s')
print(f'  Samples/second:   {train_result.metrics["train_samples_per_second"]:.2f}')

In [ ]:
# ── Cell 8: Post-training inference & comparison ─────────────────────────────
print('=== AFTER SFT ===')
response_after = generate_response(model, tokenizer, test_prompt, max_new_tokens=80)
print(response_after)

print('\n=== COMPARISON ===')
print('BEFORE:', response_before[:150])
print('AFTER: ', response_after[:150])

In [ ]:
# ── Cell 9: Evaluate on validation set ───────────────────────────────────────
eval_results = evaluate_sft(model, tokenizer, val_data, n_samples=15)
print('Eval results:', eval_results)

# Also run the trainer eval
trainer_eval = trainer.evaluate()
print('\nTrainer eval loss:', trainer_eval.get('eval_loss', 'N/A'))

In [ ]:
# ── Cell 10: Save model & training history ────────────────────────────────────
save_model(model, tokenizer, 'sft_cybersec')

# Save loss history from trainer logs
tracker = LossTracker()
for log in trainer.state.log_history:
    if 'loss' in log and 'step' in log:
        tracker.record(log['step'], log['loss'])

loss_path = str(PROJECT_ROOT / 'models' / 'sft_loss_history.json')
tracker.save(loss_path)
tracker.plot_ascii('SFT Training Loss')

print('\nNotebook 1 complete. Model saved to models/sft_cybersec')

## What just happened — key takeaways

1. **SFT is pure imitation**: the model learned to produce responses that look like the training examples. It has no concept of "quality" — just token-level probability matching.

2. **Loss on response tokens only**: the gradient doesn't flow through the prompt tokens. We're training the model to complete the prompt, not to memorize it.

3. **The limitation**: if our training data had a mix of good and bad responses, the model would imitate both equally. This is why we need DPO/RLHF — to teach the model *which responses are better*.

4. **Next notebook**: `02_dpo.ipynb` — we'll use preference pairs (chosen vs rejected) to teach quality.

### SFT loss formula recap
```
L_SFT = -1/|y| * Σ_t log P_θ(y_t | x, y_{<t})
```
where `x` = prompt tokens, `y` = response tokens, `|y|` = response length.